# ml_PCA_codes

本 notebook 为 `ml_PCA_lec.ipynb` 生成模拟数据和配图。

**数据设计**：

| 数据集 | 样本量 | 说明 | 用途 |
|---|:---:|---|---|
| 数据集 S（股票收益） | 200 × 6 | 6 支股票的模拟月度收益率，含两个潜在因子 | 贯穿全章主例 |
| 数据集 M（宏观指标） | 100 × 8 | 8 个宏观经济指标，存在明显多重共线性 | 第五部分：降维用于回归 |

**图片命名规则**：`ml_PCA_fig{NN}_{描述}.png / .svg`

| 图号 | 文件名（不含扩展名） | 布局 | figsize |
|---|---|:---:|:---:|
| 01 | `ml_PCA_fig01_raw_scatter` | 单图 | (8, 5) |
| 02 | `ml_PCA_fig02_variance_direction` | 单图 | (8, 5) |
| 03 | `ml_PCA_fig03_rotation` | 双图 | (8, 4) |
| 04 | `ml_PCA_fig04_screeplot` | 单图 | (8, 5) |
| 05 | `ml_PCA_fig05_loadings` | 单图 | (8, 5) |
| 06 | `ml_PCA_fig06_biplot` | 单图 | (8, 6) |
| 07 | `ml_PCA_fig07_scores_scatter` | 双图 | (8, 4) |
| 08 | `ml_PCA_fig08_reconstruction` | 三图 | (8, 2.8) |
| 09 | `ml_PCA_fig09_cumvar` | 单图 | (8, 5) |
| 10 | `ml_PCA_fig10_pcr_vs_ols` | 双图 | (8, 4) |


In [ ]:
# ------------------------------------------------------------
# 0. 全局设置
# ------------------------------------------------------------

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

os.makedirs('./figs', exist_ok=True)
os.makedirs('./data', exist_ok=True)

# ── 中文字体 ──
available_fonts = {f.name for f in fm.fontManager.ttflist}
font_candidates = [
    'SimHei', 'Microsoft YaHei', 'Noto Sans CJK SC',
    'Noto Sans CJK JP', 'WenQuanYi Micro Hei',
    'Arial Unicode MS', 'DejaVu Sans',
]
FONT_FAMILY = next(
    (f for f in font_candidates if f in available_fonts), 'DejaVu Sans'
)
plt.rcParams['font.sans-serif'] = [FONT_FAMILY]
plt.rcParams['axes.unicode_minus'] = False

# ── 全局基础配置（所有图通用）──
plt.rcParams.update({
    'figure.dpi':          150,
    'savefig.bbox_inches': 'tight',
    'savefig.facecolor':   'white',
    'axes.spines.top':     False,
    'axes.spines.right':   False,
    'axes.grid':           True,
    'grid.alpha':          0.22,
})

# ── 场景切换函数 ──
def fig_single():
    '''单行单图：figsize=(8,5)，字号 11pt'''
    plt.rcParams.update({
        'figure.figsize':    (8, 5),
        'font.size':         11,
        'legend.fontsize':   10,
        'axes.labelsize':    11,
        'xtick.labelsize':   10,
        'ytick.labelsize':   10,
    })

def fig_double():
    '''单行双图：figsize=(8,4)，字号 10pt'''
    plt.rcParams.update({
        'figure.figsize':    (8, 4),
        'font.size':         10,
        'legend.fontsize':   9,
        'axes.labelsize':    10,
        'xtick.labelsize':   9,
        'ytick.labelsize':   9,
    })

def fig_triple():
    '''单行三图：figsize=(8,2.8)，字号 9pt'''
    plt.rcParams.update({
        'figure.figsize':    (8, 2.8),
        'font.size':         9,
        'legend.fontsize':   8,
        'axes.labelsize':    9,
        'xtick.labelsize':   8,
        'ytick.labelsize':   8,
    })

def save_fig(fig, basename):
    fig.savefig(f'./figs/{basename}.png', dpi=150, bbox_inches='tight')
    fig.savefig(f'./figs/{basename}.svg',           bbox_inches='tight')

# 配色
C1 = '#1A3A6B'   # 深蓝
C2 = '#C8900A'   # 深黄
C3 = '#2E8B57'   # 绿
C4 = '#CC2222'   # 红
C5 = '#7B3F9E'   # 紫

RNG = np.random.default_rng(2026)
print(f'字体: {FONT_FAMILY}')

In [ ]:
# ------------------------------------------------------------
# 1. 生成数据集 S：6 支股票月度收益率（含两个潜在因子）
# ------------------------------------------------------------
# 经济背景：
#   因子 1（市场因子）：影响所有 6 支股票，但权重不同
#   因子 2（行业因子）：前 3 支是「科技股」，后 3 支是「金融股」
# 股票名称：TECH1, TECH2, TECH3, FIN1, FIN2, FIN3
# ------------------------------------------------------------

n = 200   # 200 个月（约 16 年）

# 两个潜在因子（标准正态，互相关 0.15）
cov_f = np.array([[1.0, 0.15], [0.15, 1.0]])
factors = RNG.multivariate_normal([0, 0], cov_f, size=n)
f1, f2 = factors[:, 0], factors[:, 1]   # 市场因子，行业因子

# 因子载荷矩阵（6 支股票 × 2 个因子）
# 科技股：f1 载荷高，f2 载荷为正
# 金融股：f1 载荷中等，f2 载荷为负
B = np.array([
    [0.85, 0.60],   # TECH1
    [0.80, 0.70],   # TECH2
    [0.75, 0.55],   # TECH3
    [0.70, -0.65],  # FIN1
    [0.65, -0.70],  # FIN2
    [0.60, -0.60],  # FIN3
])  # shape: (6, 2)

# 特质收益（个股特有噪声）
sigma_e = np.array([0.35, 0.38, 0.32, 0.33, 0.36, 0.34])
E = RNG.normal(0, sigma_e, size=(n, 6))

# 股票收益 = 因子贡献 + 特质收益
R = factors @ B.T + E   # shape: (n, 6)

stock_names = ['TECH1', 'TECH2', 'TECH3', 'FIN1', 'FIN2', 'FIN3']
df_S = pd.DataFrame(R, columns=stock_names)
df_S.to_csv('./data/pca_stock_returns.csv', index=False, encoding='utf-8-sig')

print('数据集 S（股票收益）描述统计：')
print(df_S.describe().round(3))
print('\n相关矩阵：')
print(df_S.corr().round(3))

In [ ]:
# ------------------------------------------------------------
# 2. 生成数据集 M：8 个宏观指标 + 目标变量（GDP 增速）
# ------------------------------------------------------------
# 经济背景：
#   8 个指标存在严重多重共线性（3 组高度相关的变量组）
#   目标变量 y = GDP 季度增速
# ------------------------------------------------------------

m = 100  # 100 个季度

# 3 个潜在因子
g1 = RNG.normal(0, 1, m)   # 增长因子
g2 = RNG.normal(0, 1, m)   # 通胀因子
g3 = RNG.normal(0, 1, m)   # 金融因子

noise = lambda s: RNG.normal(0, s, m)

macro_vars = {
    'PMI':        0.80*g1 + 0.10*g2 + noise(0.30),
    'IP_growth':  0.75*g1 + 0.15*g2 + noise(0.35),   # 工业产值增速
    'retail':     0.70*g1 + 0.20*g2 + noise(0.40),   # 零售增速
    'CPI':        0.15*g1 + 0.85*g2 + noise(0.28),
    'PPI':        0.10*g1 + 0.80*g2 + noise(0.32),   # 生产者价格
    'M2':         0.25*g1 + 0.30*g2 + 0.75*g3 + noise(0.30),
    'loan_rate':  0.05*g1 + 0.40*g2 + 0.70*g3 + noise(0.35),
    'spread':    -0.10*g1 - 0.20*g2 + 0.65*g3 + noise(0.38),  # 信用利差
}

df_M = pd.DataFrame(macro_vars)

# 目标变量：GDP 增速
y_gdp = (0.50*g1 + 0.20*g2 - 0.15*g3
         + noise(0.25))
df_M['GDP_growth'] = y_gdp

df_M.to_csv('./data/pca_macro.csv', index=False, encoding='utf-8-sig')

print('数据集 M（宏观指标）：shape =', df_M.shape)
print('\n前 5 行：')
print(df_M.head())

In [ ]:
# ------------------------------------------------------------
# 图 01：原始数据散点图（以 TECH1 vs FIN1 为例，直观展示相关结构）
# ml_PCA_fig01_raw_scatter
# ------------------------------------------------------------

fig_single()
fig, ax = plt.subplots()

# 按科技/金融分组着色
for cols, color, label in [
    (['TECH1','TECH2','TECH3'], C1, '科技股'),
    (['FIN1', 'FIN2', 'FIN3'], C2, '金融股'),
]:
    for c in cols:
        ax.scatter(df_S['TECH1'], df_S[c],
                   alpha=0.35, s=18, color=color,
                   label=label if c == cols[0] else '_')

ax.set_xlabel('TECH1 月度收益率')
ax.set_ylabel('各股票月度收益率')
ax.set_title('图 1  6 支股票收益率的两两散点（以 TECH1 为基准）\n'
             '蓝色：科技股，黄色：金融股')
ax.legend()
fig.tight_layout()
save_fig(fig, 'ml_PCA_fig01_raw_scatter')
plt.show()

In [ ]:
# ------------------------------------------------------------
# 图 02：方差最大化方向
# 取 TECH1 和 FIN1 两个变量，展示投影到不同方向的方差差异
# ml_PCA_fig02_variance_direction
# ------------------------------------------------------------

fig_single()

# 只用这两列，标准化
X2 = df_S[['TECH1', 'FIN1']].values
X2_sc = StandardScaler().fit_transform(X2)

# 求 PCA 第一主成分方向
pca2 = PCA(n_components=2)
pca2.fit(X2_sc)
pc1_dir = pca2.components_[0]   # 第一主成分方向向量
pc2_dir = pca2.components_[1]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

for ax, direction, color, title_suffix in [
    (axes[0], np.array([1, 0]),  C2,
     '任意方向（仅 $x_1$ 轴）\n投影方差较小'),
    (axes[1], pc1_dir,           C1,
     '第一主成分方向\n投影方差最大'),
]:
    ax.scatter(X2_sc[:, 0], X2_sc[:, 1],
               alpha=0.35, s=18, color='#888888')

    # 方向箭头
    scale = 2.5
    ax.annotate('', xy=scale*direction, xytext=-scale*direction,
                arrowprops=dict(arrowstyle='->', color=color, lw=2.2))

    # 投影点
    proj = (X2_sc @ direction)[:, None] * direction
    ax.scatter(proj[:, 0], proj[:, 1],
               alpha=0.55, s=12, color=color)

    # 投影线（灰色细线）
    for i in range(0, len(X2_sc), 3):
        ax.plot([X2_sc[i,0], proj[i,0]], [X2_sc[i,1], proj[i,1]],
                color='#aaaaaa', lw=0.5, alpha=0.6)

    proj_var = np.var(X2_sc @ direction)
    ax.set_title(f'{title_suffix}\n投影方差 = {proj_var:.3f}',
                 fontsize=10)
    ax.set_xlabel('TECH1（标准化）')
    ax.set_ylabel('FIN1（标准化）')
    ax.set_xlim(-3.2, 3.2)
    ax.set_ylim(-3.2, 3.2)
    ax.set_aspect('equal')

fig.suptitle('图 2  方差最大化方向：PCA 选择使投影方差最大的轴',
             fontsize=11, y=1.01)
fig.tight_layout()
save_fig(fig, 'ml_PCA_fig02_variance_direction')
plt.show()

In [ ]:
# ------------------------------------------------------------
# 图 03：旋转坐标系（原始坐标 vs 主成分坐标）
# ml_PCA_fig03_rotation
# ------------------------------------------------------------

fig_double()

fig, axes = plt.subplots(1, 2)

# 左图：原始标准化空间
ax = axes[0]
ax.scatter(X2_sc[:, 0], X2_sc[:, 1],
           alpha=0.45, s=20, color='#888888')

# 画主成分方向（红、蓝箭头）
scale = 2.2
for direction, color, label in [
    (pca2.components_[0], C1, f'PC1 (解释方差 {pca2.explained_variance_ratio_[0]:.1%})'),
    (pca2.components_[1], C2, f'PC2 (解释方差 {pca2.explained_variance_ratio_[1]:.1%})'),
]:
    ax.annotate('', xy=scale*direction, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.0))
    ax.text(*(scale*direction*1.12), label, color=color, fontsize=8.5)

ax.axhline(0, color='#cccccc', lw=0.8)
ax.axvline(0, color='#cccccc', lw=0.8)
ax.set_xlabel('TECH1（标准化）')
ax.set_ylabel('FIN1（标准化）')
ax.set_title('(a) 原始坐标系\n主成分方向（箭头）')
ax.set_aspect('equal')

# 右图：主成分坐标系
ax = axes[1]
scores2 = pca2.transform(X2_sc)
ax.scatter(scores2[:, 0], scores2[:, 1],
           alpha=0.45, s=20, color='#888888')
ax.axhline(0, color='#cccccc', lw=0.8)
ax.axvline(0, color='#cccccc', lw=0.8)
ax.set_xlabel(f'PC1（方差 {pca2.explained_variance_ratio_[0]:.1%}）')
ax.set_ylabel(f'PC2（方差 {pca2.explained_variance_ratio_[1]:.1%}）')
ax.set_title('(b) 主成分坐标系\n两轴不相关，方差沿 PC1 最大')
ax.set_aspect('equal')

fig.suptitle('图 3  坐标系旋转：PCA 将原始坐标轴旋转为互不相关的主成分轴',
             fontsize=10, y=1.01)
fig.tight_layout()
save_fig(fig, 'ml_PCA_fig03_rotation')
plt.show()

print(f'PC1 方向: {pca2.components_[0].round(4)}')
print(f'PC2 方向: {pca2.components_[1].round(4)}')
print(f'PC1、PC2 得分的相关系数: {np.corrcoef(scores2[:,0], scores2[:,1])[0,1]:.6f}')

In [ ]:
# ------------------------------------------------------------
# 3. 对 6 维股票收益数据做完整 PCA（标准化后）
# ------------------------------------------------------------

X6 = df_S.values
scaler6 = StandardScaler()
X6_sc = scaler6.fit_transform(X6)

pca6 = PCA(n_components=6)
pca6.fit(X6_sc)

scores6 = pca6.transform(X6_sc)   # (200, 6)

print('各主成分解释方差比：')
for i, r in enumerate(pca6.explained_variance_ratio_):
    print(f'  PC{i+1}: {r:.4f}  ({r:.1%})')
print(f'\nPC1+PC2 累计解释方差: {pca6.explained_variance_ratio_[:2].sum():.1%}')

# 因子载荷矩阵（每列是一个主成分在各原始变量上的载荷）
loadings = pca6.components_.T  # shape: (6 vars, 6 PCs)
df_loadings = pd.DataFrame(
    loadings,
    index=stock_names,
    columns=[f'PC{i+1}' for i in range(6)]
)
print('\n因子载荷矩阵（前 3 个主成分）：')
print(df_loadings.iloc[:, :3].round(3))

In [ ]:
# ------------------------------------------------------------
# 图 04：碎石图（Scree Plot）
# ml_PCA_fig04_screeplot
# ------------------------------------------------------------

fig_single()

evr = pca6.explained_variance_ratio_
cumvar = np.cumsum(evr)
pcs = np.arange(1, 7)

fig, ax1 = plt.subplots()

# 柱状：各主成分解释方差比
bars = ax1.bar(pcs, evr * 100, color=C1, alpha=0.75,
               width=0.5, label='各 PC 解释方差 (%)')
for bar, val in zip(bars, evr):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.8,
             f'{val:.1%}', ha='center', fontsize=9.5)

# 折线：累计解释方差比（右轴）
ax2 = ax1.twinx()
ax2.plot(pcs, cumvar * 100, 'o-',
         color=C2, lw=2.0, ms=6, label='累计解释方差 (%)')
ax2.axhline(80, color='#aaaaaa', lw=1.0, linestyle=':')
ax2.text(6.15, 80.5, '80%', color='#888888', fontsize=9)
ax2.set_ylim(0, 108)
ax2.set_ylabel('累计解释方差 (%)')
ax2.spines['top'].set_visible(False)

# 标注「肘部」（PC2 之后斜率明显减缓）
ax1.axvline(2.5, color=C4, lw=1.2, linestyle='--', alpha=0.7)
ax1.text(2.55, evr.max()*100*0.88,
         '← 选取 2 个主成分\n（肘部）',
         color=C4, fontsize=9)

ax1.set_xlabel('主成分编号')
ax1.set_ylabel('解释方差比 (%)')
ax1.set_xticks(pcs)
ax1.set_xticklabels([f'PC{i}' for i in pcs])
ax1.set_title('图 4  碎石图：各主成分的解释方差与累计解释方差')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           loc='center right', fontsize=9)
fig.tight_layout()
save_fig(fig, 'ml_PCA_fig04_screeplot')
plt.show()

In [ ]:
# ------------------------------------------------------------
# 图 05：因子载荷热力图
# ml_PCA_fig05_loadings
# ------------------------------------------------------------

fig_single()

load2 = df_loadings.iloc[:, :2]  # 只看前两个主成分

fig, ax = plt.subplots(figsize=(6, 4.5))

import matplotlib.colors as mcolors
cmap = plt.cm.RdBu_r
im = ax.imshow(load2.values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')

# 在每个格子中显示数值
for i in range(load2.shape[0]):
    for j in range(load2.shape[1]):
        val = load2.iloc[i, j]
        color = 'white' if abs(val) > 0.55 else 'black'
        ax.text(j, i, f'{val:.3f}',
                ha='center', va='center',
                fontsize=11, color=color, fontweight='bold')

ax.set_xticks([0, 1])
ax.set_xticklabels(['PC1（市场因子）', 'PC2（行业因子）'])
ax.set_yticks(range(6))
ax.set_yticklabels(stock_names)
plt.colorbar(im, ax=ax, label='载荷值')
ax.set_title('图 5  因子载荷热力图（前两个主成分）\n'
             '红色=正载荷，蓝色=负载荷')
ax.grid(False)
fig.tight_layout()
save_fig(fig, 'ml_PCA_fig05_loadings')
plt.show()

In [ ]:
# ------------------------------------------------------------
# 图 06：双标图（Biplot）
# ml_PCA_fig06_biplot
# ------------------------------------------------------------

plt.rcParams.update({
    'figure.figsize': (8, 6),
    'font.size': 11,
    'legend.fontsize': 10,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

fig, ax = plt.subplots()

# 主成分得分（只取前 2 个 PC）
sc = scores6[:, :2]
# 标准化得分（让散点和箭头在同一尺度下可比）
sc_norm = sc / sc.std(axis=0)

ax.scatter(sc_norm[:, 0], sc_norm[:, 1],
           alpha=0.30, s=16, color='#999999', zorder=1)

# 载荷箭头（放大到合适尺度）
scale_arrow = 3.0
for i, name in enumerate(stock_names):
    lx = loadings[i, 0] * scale_arrow
    ly = loadings[i, 1] * scale_arrow
    color = C1 if name.startswith('TECH') else C2
    ax.annotate('', xy=(lx, ly), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color,
                                lw=2.0, mutation_scale=14),
                zorder=3)
    offset = np.array([lx, ly]) / np.linalg.norm([lx, ly]) * 0.22
    ax.text(lx + offset[0], ly + offset[1], name,
            color=color, fontsize=10.5, fontweight='bold',
            ha='center', va='center', zorder=4)

ax.axhline(0, color='#dddddd', lw=0.8)
ax.axvline(0, color='#dddddd', lw=0.8)

patches = [
    mpatches.Patch(color=C1, label='科技股'),
    mpatches.Patch(color=C2, label='金融股'),
]
ax.legend(handles=patches, loc='lower right')

evr2 = pca6.explained_variance_ratio_
ax.set_xlabel(f'PC1（解释方差 {evr2[0]:.1%}）')
ax.set_ylabel(f'PC2（解释方差 {evr2[1]:.1%}）')
ax.set_title('图 6  双标图（Biplot）\n'
             '散点 = 样本得分，箭头 = 各股票在主成分空间的载荷')
fig.tight_layout()
save_fig(fig, 'ml_PCA_fig06_biplot')
plt.show()

In [ ]:
# ------------------------------------------------------------
# 图 07：主成分得分散点图（PC1 vs PC2，高亮极端月份）
# ml_PCA_fig07_scores_scatter
# ------------------------------------------------------------

fig_double()

fig, axes = plt.subplots(1, 2)

# 左图：PC1 vs PC2 得分
ax = axes[0]
s1, s2 = scores6[:, 0], scores6[:, 1]

# 按 PC1 分位数着色（代表市场整体走势强弱）
q_low  = np.percentile(s1, 15)
q_high = np.percentile(s1, 85)
mask_low  = s1 < q_low
mask_high = s1 > q_high
mask_mid  = ~mask_low & ~mask_high

ax.scatter(s1[mask_mid],  s2[mask_mid],
           alpha=0.40, s=16, color='#aaaaaa', label='普通月份')
ax.scatter(s1[mask_high], s2[mask_high],
           alpha=0.80, s=28, color=C2, label='市场强势月（PC1 高）')
ax.scatter(s1[mask_low],  s2[mask_low],
           alpha=0.80, s=28, color=C4, label='市场弱势月（PC1 低）')
ax.axhline(0, color='#dddddd', lw=0.8)
ax.axvline(0, color='#dddddd', lw=0.8)
ax.set_xlabel('PC1 得分（市场因子）')
ax.set_ylabel('PC2 得分（行业因子）')
ax.set_title('(a) PC1 vs PC2 得分散点图\n高亮极端市场状态')
ax.legend(fontsize=8)

# 右图：PC1 得分的时序图（展示市场状态的时序变化）
ax2 = axes[1]
ax2.plot(range(n), s1, color='#888888', lw=0.8, alpha=0.7)
ax2.fill_between(range(n), s1, 0,
                 where=(s1 > 0), alpha=0.35, color=C2, label='正值（市场强势）')
ax2.fill_between(range(n), s1, 0,
                 where=(s1 < 0), alpha=0.35, color=C4, label='负值（市场弱势）')
ax2.axhline(0, color='#333333', lw=0.8)
ax2.set_xlabel('时间（月）')
ax2.set_ylabel('PC1 得分')
ax2.set_title('(b) PC1 得分的时序演变\n类似「市场指数」的波动模式')
ax2.legend(fontsize=8)

fig.suptitle('图 7  主成分得分：PC1 捕捉市场整体走势，PC2 区分科技/金融',
             fontsize=10, y=1.02)
fig.tight_layout()
save_fig(fig, 'ml_PCA_fig07_scores_scatter')
plt.show()

In [ ]:
# ------------------------------------------------------------
# 图 08：用不同数量主成分重构原始数据
# ml_PCA_fig08_reconstruction
# ------------------------------------------------------------

fig_triple()

# 用 k 个主成分重构
def reconstruct(X_sc, pca, k):
    scores_k = pca.transform(X_sc)[:, :k]
    return scores_k @ pca.components_[:k, :]

fig, axes = plt.subplots(1, 3)

for ax, k in zip(axes, [1, 2, 6]):
    X_rec = reconstruct(X6_sc, pca6, k)
    # 计算重构误差（MSE per sample）
    rec_err = np.mean((X6_sc - X_rec)**2, axis=1)
    cum_var = pca6.explained_variance_ratio_[:k].sum()

    ax.scatter(X6_sc[:, 0], X_rec[:, 0],
               alpha=0.35, s=10, color=C1)
    # 完美重构参考线
    lim = max(abs(X6_sc[:, 0]).max(), abs(X_rec[:, 0]).max()) * 1.05
    ax.plot([-lim, lim], [-lim, lim],
            color='#aaaaaa', lw=1.0, linestyle='--')
    ax.set_xlabel('原始值')
    ax.set_ylabel('重构值')
    ax.set_title(f'k={k} 个 PC\n累计方差 {cum_var:.0%}\n'
                 f'MSE={rec_err.mean():.3f}')
    ax.set_aspect('equal')

fig.suptitle('图 8  不同主成分数量的重构效果（以 TECH1 为例）',
             fontsize=9, y=1.02)
fig.tight_layout()
save_fig(fig, 'ml_PCA_fig08_reconstruction')
plt.show()

In [ ]:
# ------------------------------------------------------------
# 图 09：宏观数据集 M 的 PCA 累计方差
# ml_PCA_fig09_cumvar
# ------------------------------------------------------------

fig_single()

macro_cols = [c for c in df_M.columns if c != 'GDP_growth']
X_M = df_M[macro_cols].values
X_M_sc = StandardScaler().fit_transform(X_M)

pca_M = PCA(n_components=8)
pca_M.fit(X_M_sc)
evr_M = pca_M.explained_variance_ratio_
cumvar_M = np.cumsum(evr_M)
pcs_M = np.arange(1, 9)

fig, ax = plt.subplots()
ax.bar(pcs_M, evr_M * 100,
       color=C3, alpha=0.75, width=0.5, label='各 PC 解释方差 (%)')
ax2 = ax.twinx()
ax2.plot(pcs_M, cumvar_M * 100, 'o-',
         color=C2, lw=2.0, ms=6, label='累计解释方差 (%)')
for k_thresh, ls in [(80, ':'), (90, '--')]:
    ax2.axhline(k_thresh, color='#aaaaaa', lw=0.9, linestyle=ls)
    ax2.text(8.15, k_thresh+0.5, f'{k_thresh}%',
             color='#888888', fontsize=9)
ax2.set_ylim(0, 108)
ax2.set_ylabel('累计解释方差 (%)')
ax2.spines['top'].set_visible(False)
ax.set_xlabel('主成分编号')
ax.set_ylabel('解释方差比 (%)')
ax.set_xticks(pcs_M)
ax.set_xticklabels([f'PC{i}' for i in pcs_M])
ax.set_title('图 9  宏观指标数据集的碎石图\n'
             '前 3 个主成分累计解释方差超过 85%')
lines1, lb1 = ax.get_legend_handles_labels()
lines2, lb2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, lb1+lb2, fontsize=9, loc='center right')
fig.tight_layout()
save_fig(fig, 'ml_PCA_fig09_cumvar')
plt.show()

print('宏观数据前 4 个主成分的累计解释方差：')
for k in [1, 2, 3, 4]:
    print(f'  前 {k} 个 PC: {cumvar_M[k-1]:.1%}')

In [ ]:
# ------------------------------------------------------------
# 图 10：主成分回归（PCR）vs OLS
# 以宏观数据集预测 GDP 增速为例
# ml_PCA_fig10_pcr_vs_ols
# ------------------------------------------------------------

fig_double()

y_M = df_M['GDP_growth'].values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# OLS（含全部 8 个宏观指标）
ols = Pipeline([('sc', StandardScaler()), ('reg', LinearRegression())])
ols_cv = -cross_val_score(ols, X_M, y_M, cv=kf,
                           scoring='neg_mean_squared_error')

# PCR：遍历不同主成分数量
pcr_cv_list = []
k_vals = range(1, 9)
for k in k_vals:
    pipe_pcr = Pipeline([
        ('sc',  StandardScaler()),
        ('pca', PCA(n_components=k)),
        ('reg', LinearRegression()),
    ])
    cv_scores = -cross_val_score(
        pipe_pcr, X_M, y_M, cv=kf,
        scoring='neg_mean_squared_error'
    )
    pcr_cv_list.append(cv_scores.mean())

best_k = int(np.argmin(pcr_cv_list)) + 1

fig, axes = plt.subplots(1, 2)

# 左图：CV MSE vs 主成分数量
ax = axes[0]
ax.plot(list(k_vals), pcr_cv_list, 'o-',
        color=C1, lw=1.8, ms=5, label='PCR（不同 k）')
ax.axhline(ols_cv.mean(), color=C2, lw=1.6,
           linestyle='--', label=f'OLS 全变量 ({ols_cv.mean():.4f})')
ax.axvline(best_k, color=C4, lw=1.2, linestyle=':',
           label=f'最优 k={best_k}')
ax.scatter([best_k], [pcr_cv_list[best_k-1]],
           color=C4, s=60, zorder=5)
ax.set_xlabel('主成分数量 k')
ax.set_ylabel('5 折 CV MSE')
ax.set_title('(a) PCR vs OLS：交叉验证误差\n最优主成分数量的选择')
ax.set_xticks(list(k_vals))
ax.legend(fontsize=8)

# 右图：最优 PCR 的预测值 vs 真实值
ax2 = axes[1]
pipe_best = Pipeline([
    ('sc',  StandardScaler()),
    ('pca', PCA(n_components=best_k)),
    ('reg', LinearRegression()),
])
pipe_best.fit(X_M, y_M)
y_pred = pipe_best.predict(X_M)

ax2.scatter(y_M, y_pred, alpha=0.45, s=20, color=C3)
lim = max(abs(y_M).max(), abs(y_pred).max()) * 1.05
ax2.plot([-lim, lim], [-lim, lim],
         color='#aaaaaa', lw=1.0, linestyle='--')
r2 = pipe_best.score(X_M, y_M)
ax2.set_xlabel('真实 GDP 增速')
ax2.set_ylabel('PCR 预测值')
ax2.set_title(f'(b) 最优 PCR（k={best_k}）拟合效果\n$R^2$ = {r2:.3f}')

fig.suptitle('图 10  主成分回归（PCR）：用主成分替代原始高度相关变量',
             fontsize=10, y=1.02)
fig.tight_layout()
save_fig(fig, 'ml_PCA_fig10_pcr_vs_ols')
plt.show()

print(f'最优主成分数: k={best_k}')
print(f'PCR  CV MSE: {pcr_cv_list[best_k-1]:.4f}')
print(f'OLS  CV MSE: {ols_cv.mean():.4f}')